In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
def create_features(deop, solcast, steps_per_day=288):
    df = pd.DataFrame(index=deop.index)
    
    # Weather
    df['gti_target'] = solcast['gti']
    df['cloud_opacity_target'] = solcast['cloud_opacity']
    df['snow_depth'] = solcast['snow_depth']
    
    # Power Lag
    df['lag_1d'] = deop['power-gen-pv-ave'].shift(steps_per_day)
    df['lag_2d'] = deop['power-gen-pv-ave'].shift(steps_per_day * 2)
    df['lag_3d'] = deop['power-gen-pv-ave'].shift(steps_per_day * 3)
    df['lag_7d'] = deop['power-gen-pv-ave'].shift(steps_per_day * 7)
    df['lag_14d'] = deop['power-gen-pv-ave'].shift(steps_per_day * 14)

    # Volatility & Trend
    df['power_volatility_1d'] = deop['power-gen-pv-ave'].shift(steps_per_day).rolling(24).std()
    df['power_trend_1d'] = df['lag_1d'] - df['lag_2d']
    df['gti_volatility_1d'] = solcast['gti'].shift(steps_per_day).rolling(24).std()
    
    # Peak of previous day
    daily_max = deop['power-gen-pv-ave'].resample('D').max().shift(1)
    df['yesterday_peak'] = df.index.normalize().map(daily_max)

    # Geometry & Time
    hour_num = df.index.hour + df.index.minute / 60
    df['solar_potential'] = np.maximum(0, np.sin(np.pi * (hour_num - 6) / 12))
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

    return df.dropna()

In [ ]:
def train_and_test_xgboost(deop, solcast, deop2022, solcast2022, features):

    X_train_full = create_features(deop, solcast)[features]
    X_test_2022 = create_features(deop2022, solcast2022)[features]

    y_train_full = deop.loc[X_train_full.index, 'power-gen-pv-ave']
    y_2022_full = deop2022.loc[X_test_2022.index, 'power-gen-pv-ave']

    mask_2022 = (X_test_2022.index.hour >= 4) & (X_test_2022.index.hour < 21)

    # Split (90% Train / 10% Val)
    split_idx = int(len(X_train_full) * 0.90) 
    X_train, y_train = X_train_full.iloc[:split_idx], y_train_full.iloc[:split_idx]
    X_val, y_val = X_train_full.iloc[split_idx:], y_train_full.iloc[split_idx:]

    model = xgb.XGBRegressor(
        objective='reg:tweedie', 
        tweedie_variance_power=1.5, 
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        reg_lambda=0.5,
        subsample=0.9,
        colsample_bytree=0.8,
        random_state=42,
        tree_method='hist',
        early_stopping_rounds=150
    )

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=200)

    # Predictions
    raw_preds = model.predict(X_test_2022)
    final_preds_series = pd.Series(np.where(mask_2022, np.maximum(0, raw_preds), 0), index=X_test_2022.index)

    r2_5min = r2_score(y_2022_full, final_preds_series)
    mae_5min = mean_absolute_error(y_2022_full, final_preds_series)

    y_hourly_actual = y_2022_full.resample('1h').mean()
    y_hourly_preds = final_preds_series.resample('1h').mean()
    r2_hourly = r2_score(y_hourly_actual, y_hourly_preds)
    mae_hourly = mean_absolute_error(y_hourly_actual, y_hourly_preds)

    y_daily_actual = y_2022_full.resample('1D').mean()
    y_daily_preds = final_preds_series.resample('1D').mean()
    r2_daily = r2_score(y_daily_actual, y_daily_preds)
    mae_daily = mean_absolute_error(y_daily_actual, y_daily_preds)

    print(f"5-Minute R2 Score:  {r2_5min:.4f}")
    print(f"Hourly R2 Score:    {r2_hourly:.4f}")
    print(f"Daily R2 Score:     {r2_daily:.4f}")
    print(f"5-Minute MAE:       {mae_5min:.2f} kW")
    print(f"Hourly MAE:         {mae_hourly:.2f} kW")
    print(f"Daily MAE:          {mae_daily:.2f} kW")
    print(f"Max Predicted Peak: {final_preds_series.max():.2f} kW")
    return final_preds_series